# MobileNetV2 transfer - test run


In [ ]:
import os
from pathlib import Path
from azure.storage.blob import BlobServiceClient

conn = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
if conn:
    service = BlobServiceClient.from_connection_string(conn)
    container = service.get_container_client("kcrowly-vast")
    local_data = ROOT / "Data" / "raw"
    if not local_data.exists() or not any(local_data.iterdir()):
        print("Downloading Data/raw from Azure...")
        for blob in container.list_blobs():
            name = blob.name
            if not name.startswith("Data/raw/"):
                continue
            rel = name[len("Data/raw/"):]
            if not rel:
                continue
            dest = local_data / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(dest, "wb") as f:
                container.download_blob(name).readinto(f)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import mlflow
import mlflow.tensorflow
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

from neurovision.data import NUM_CLASSES, DEFAULT_SIZE, make_augmentation, make_datasets

mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_experiment("NeuroVision")

print("root:", ROOT)

Load the data
MobileNetV2 preprocessing maps pixels to [-1, 1].

In [ ]:
datasets = make_datasets(
    str(ROOT / "Data" / "raw"),
    size=DEFAULT_SIZE,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    augmentation=make_augmentation(rotation=0.03, zoom=0.1, translation=0.1),
    preprocess="mobilenet",
    cache=False,
)

train_ds = datasets["train"]
val_ds = datasets["val"]
test_ds = datasets["test"]

print("batches:", len(train_ds), len(val_ds), len(test_ds))

Build the model
Frozen ImageNet base, new classification head.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)
base.trainable = False

model = tf.keras.Sequential(
    [
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="mobilenetv2_transfer",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("params:", model.count_params())

Train, log metrics, save artifacts

In [ ]:
OUT = ROOT / "models" / "mobilenetv2-transfer-test"
OUT.mkdir(parents=True, exist_ok=True)

with mlflow.start_run(run_name="mobilenetv2-transfer-test") as run:
    mlflow.log_params({
        "model": "mobilenetv2",
        "strategy": "transfer",
        "preprocess": "mobilenet",
        "learning_rate": 1e-3,
        "batch_size": 32,
        "epochs": 2,
        "total_params": model.count_params(),
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=2,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(OUT / "best.keras"),
                monitor="val_loss",
                save_best_only=True,
            ),
            tf.keras.callbacks.BackupAndRestore(backup_dir=str(OUT / "backup")),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=2, restore_best_weights=True
            ),
        ],
    )

    test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
    y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
    y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

    mlflow.log_metrics({
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "test_f1_macro": f1_score(y_true, y_pred, average="macro"),
    })

    mlflow.tensorflow.log_model(model, name="model")
    mlflow.log_artifact(str(OUT / "best.keras"))

print("run id:", run.info.run_id)

## Upload model to Azure Blob Storage


In [ ]:
import os
from azure.storage.blob import BlobServiceClient

conn = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
if not conn:
    raise RuntimeError("Set AZURE_STORAGE_CONNECTION_STRING env var before running this cell")

service = BlobServiceClient.from_connection_string(conn)
container = service.get_container_client("kcrowly-vast")

# Upload the best model from this run
best_model = OUT / "best.keras"
if best_model.exists():
    blob_name = f"models/{best_model.parent.name}/{best_model.name}"
    with open(best_model, "rb") as f:
        container.upload_blob(blob_name, f, overwrite=True)
    print(f"Uploaded -> {blob_name}")
else:
    print("No best.keras found")